# Group23: Analysis from saved metric tables

Weights and bulk PNGs are omitted. Model, inference and complete-run plotting code remains in src/. Custom outputs are written to analysis_outputs/.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT = Path.cwd()
sys.path.insert(0, str(PROJECT / 'src'))
ROOT = PROJECT / 'results'
TABLES = ROOT / 'analysis/tables'
CUSTOM = PROJECT / 'analysis_outputs'
CUSTOM.mkdir(parents=True, exist_ok=True)
if not TABLES.exists():
    raise FileNotFoundError('Open this notebook from the project folder with retained CSV tables.')

## 1. Summary and matching settings
Patient macro metrics are primary. Check the settings comparison before claiming a controlled experiment. Loss values from different loss objectives are not directly comparable.

In [ ]:
display(pd.read_csv(TABLES / 'dataset_summary.csv'))
display(pd.read_csv(TABLES / 'model_comparison.csv'))
display(pd.read_csv(TABLES / 'comparison_settings.csv'))

## 2. The same patients across models
Slice scores within a patient are correlated. Patient-level comparisons provide a more appropriate summary of differences.

In [ ]:
patients = pd.read_csv(TABLES / 'all_patient_metrics.csv')
slices = pd.read_csv(TABLES / 'all_slice_metrics.csv')
display(patients.groupby('model').dice.agg(['count','mean','median','min','max']))
paired = patients.pivot(index='patient', columns='model', values='dice')
display(paired)
paired.to_csv(CUSTOM / 'patient_comparison.csv')

## 3. Lesion size, false positives and uncertainty
Area thresholds are defined from training masks. Empty-mask error rates count any predicted foreground. Bootstrap intervals resample patients; interpret them cautiously, particularly with few patients.

In [ ]:
for name in ['lesion_size.csv','empty_mask_errors.csv','patient_bootstrap.csv']:
    display(pd.read_csv(TABLES / name))

## Lowest-overlap positive case
Inspect the same slice using both models' metric rows. Bulk image files are omitted.

In [ ]:
MODEL = 'unet'
row = slices[(slices.model == MODEL) & (slices.true_pixels > 0)].sort_values('dice').iloc[0]
PATIENT, SLICE = row.patient, int(row.slice_id)
display(slices[(slices.patient == PATIENT) & (slices.slice_id == SLICE)])

## 5. Extension example: predicted area error
This analysis uses existing predictions. Add other groupings or plots here; save custom outputs separately so rebuilding standard figures does not overwrite them.

In [ ]:
area = slices.groupby('model').area_error.agg(['mean','median','min','max'])
display(area)
area.to_csv(CUSTOM / 'area_error_summary.csv')
fig, ax = plt.subplots(figsize=(8,4))
for name, part in slices.groupby('model'):
    ax.hist(part.area_error, bins=30, alpha=.45, label=name)
ax.set(xlabel='Predicted minus true foreground pixels', ylabel='Slices'); ax.legend()
plt.tight_layout(); fig.savefig(CUSTOM / 'area_error_histogram.png', dpi=180); plt.show()